# Multi-Class Multi-Label Classification on Crabs

This notebook demonstrates a multi-class multi-label classification task in `sltn` using first-order logic and neural predicate grounding.

## Problem

Each Leptograpsus crab is described by five numerical measurements:

- frontal lobe size (`FL`);
- rear width (`RW`);
- carapace length (`CL`);
- carapace width (`CW`);
- body depth (`BD`).

Each crab has two independent categorical attributes:

- color: blue or orange;
- sex: male or female.

The task is represented using four labels:

- $L_{\mathrm{blue}}$;
- $L_{\mathrm{orange}}$;
- $L_{\mathrm{male}}$;
- $L_{\mathrm{female}}$.

A neural function $f$ maps each crab sample to four label-specific truth values. The predicate $P(f(x), L)$ then extracts the truth value associated with label $L$ for sample $x$.

Unlike standard single-label multi-class classification over one label set, this task involves two independent attribute groups. For that reason, the model uses sigmoid activations rather than a single softmax over all four labels.

The training knowledge base contains six axioms:

$$
\begin{aligned}
&\forall x_{\mathrm{blue}} \; P(f(x_{\mathrm{blue}}), L_{\mathrm{blue}}), \\
&\forall x_{\mathrm{orange}} \; P(f(x_{\mathrm{orange}}), L_{\mathrm{orange}}), \\
&\forall x_{\mathrm{male}} \; P(f(x_{\mathrm{male}}), L_{\mathrm{male}}), \\
&\forall x_{\mathrm{female}} \; P(f(x_{\mathrm{female}}), L_{\mathrm{female}}), \\
&\forall x \; \lnot \bigl(P(f(x), L_{\mathrm{blue}}) \land P(f(x), L_{\mathrm{orange}})\bigr), \\
&\forall x \; \lnot \bigl(P(f(x), L_{\mathrm{male}}) \land P(f(x), L_{\mathrm{female}})\bigr).
\end{aligned}
$$

The first four axioms provide supervised evidence for the four labels. The last two axioms encode mutual exclusion within each attribute group: a crab should not be both blue and orange, and should not be both male and female. After training, three query formulas are evaluated on the test set:

$$
\phi_1 : \forall x \; \bigl(P(f(x), L_{\mathrm{blue}}) \to \lnot P(f(x), L_{\mathrm{orange}})\bigr),
$$

$$
\phi_2 : \forall x \; \bigl(P(f(x), L_{\mathrm{blue}}) \to P(f(x), L_{\mathrm{orange}})\bigr),
$$

$$
\phi_3 : \forall x \; \bigl(P(f(x), L_{\mathrm{blue}}) \to P(f(x), L_{\mathrm{male}})\bigr).
$$

These formulas are not included in the training knowledge base. They are used to inspect whether the learned model satisfies expected consequences, contradictions, or unrelated implications.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import torch
import numpy as np

torch.manual_seed(0)
np.random.seed(0)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## Dataset: Crabs

The Crabs dataset contains morphological measurements and two labels for each sample: color and sex.

The dataset is loaded from a local file, shuffled, encoded numerically, and split into training and test subsets. The five measurements are used as input features. The color and sex labels are kept separately because they correspond to two independent prediction targets.

The following cell loads the dataset, converts categorical labels to integer codes, extracts the five numerical features, and creates training and test splits.

In [ ]:
import pandas as pd

df = pd.read_csv("../datasets/crabs.dat", sep=" ", skipinitialspace=True)
df = df.sample(frac=1)   # shuffle
df = df.replace({'B': 0, 'O': 1, 'M': 2, 'F': 3})

features = torch.tensor(df[['FL', 'RW', 'CL', 'CW', 'BD']].to_numpy()).float()
labels_sex   = torch.tensor(df['sex']).long()
labels_color = torch.tensor(df['sp']).long()

train_data        = features[:160]
test_data         = features[160:]
train_sex_labels  = labels_sex[:160]
test_sex_labels   = labels_sex[160:]
train_color_labels = labels_color[:160]
test_color_labels  = labels_color[160:]

## 1. Signature

The signature declares the symbolic vocabulary used to express the learning problem.

This notebook uses two sorts:

- `Item`, representing a crab sample with five numerical features;
- `Label`, representing one of the four label constants.

It also declares:

- a function `f : Item → Label`, grounded by a neural network that maps a crab sample to a four-dimensional vector of label truth values;
- a predicate `P : Label × Label → Bool`, used to select the truth value associated with a particular label;
- four label constants: `L_blue`, `L_orange`, `L_male`, and `L_female`;
- variables for all samples and for class-specific subsets.

In [ ]:
import sltn
from sltn import Signature, KB, kb_describe, Interpretation, Logic
import sltn.interpretation.operators.aggregator as agg


# Declare signature
sig  = Signature("crabs_clf")

# sorts
sig.sort("Item")   # crab sample
sig.sort("Label")  # one-hot label

# functions
sig.function("f", ["Item"], "Label")

# predicates
sig.predicate("P", ["Label", "Label"])

# constants
for name in ["L_blue", "L_orange", "L_male", "L_female"]:
    sig.constant(name, "Label")

# variables
sig.variable("x",        "Item")
sig.variable("x_blue",   "Item")
sig.variable("x_orange", "Item")
sig.variable("x_male",   "Item")
sig.variable("x_female", "Item")

print(sig)


## 2. Formulas

The training knowledge base consists of six first-order formulas. The four supervised formulas state that examples belonging to a given class should satisfy the corresponding label predicate:

$$
\forall x_{\mathrm{blue}} \; P(f(x_{\mathrm{blue}}), L_{\mathrm{blue}}),
$$

$$
\forall x_{\mathrm{orange}} \; P(f(x_{\mathrm{orange}}), L_{\mathrm{orange}}),
$$

$$
\forall x_{\mathrm{male}} \; P(f(x_{\mathrm{male}}), L_{\mathrm{male}}),
$$

$$
\forall x_{\mathrm{female}} \; P(f(x_{\mathrm{female}}), L_{\mathrm{female}}).
$$

The two mutual-exclusion formulas state that the two labels within each attribute group should not hold simultaneously:

$$
\forall x \; \lnot \bigl(P(f(x), L_{\mathrm{blue}}) \land P(f(x), L_{\mathrm{orange}})\bigr),
$$

$$
\forall x \; \lnot \bigl(P(f(x), L_{\mathrm{male}}) \land P(f(x), L_{\mathrm{female}})\bigr).
$$

We also define three query formulas that are evaluated after training but are not part of the knowledge base:

$$
\phi_1 : P(f(x), L_{\mathrm{blue}}) \to \lnot P(f(x), L_{\mathrm{orange}}),
$$

$$
\phi_2 : P(f(x), L_{\mathrm{blue}}) \to P(f(x), L_{\mathrm{orange}}),
$$

$$
\phi_3 : P(f(x), L_{\mathrm{blue}}) \to P(f(x), L_{\mathrm{male}}).
$$

During evaluation, each query is universally quantified over the test samples.

In [ ]:
# --- Knowledge base ---
phi_blue   = sig.parse("∀ x_blue:   P(f(x_blue),   L_blue)")
phi_orange = sig.parse("∀ x_orange: P(f(x_orange), L_orange)")
phi_male   = sig.parse("∀ x_male:   P(f(x_male),   L_male)")
phi_female = sig.parse("∀ x_female: P(f(x_female), L_female)")
phi_mutex_color = sig.parse("∀ x: ¬(P(f(x), L_blue) & P(f(x), L_orange))")
phi_mutex_sex   = sig.parse("∀ x: ¬(P(f(x), L_male) & P(f(x), L_female))")

phi_kb = KB([phi_blue, phi_orange, phi_male, phi_female,
                 phi_mutex_color, phi_mutex_sex])

# --- Query formulas (not in KB) ---
phi1_inner = sig.parse("(P(f(x), L_blue) -> ¬ P(f(x), L_orange))")
phi2_inner = sig.parse("(P(f(x), L_blue) -> P(f(x), L_orange))")
phi3_inner = sig.parse("(P(f(x), L_blue) -> P(f(x), L_male))")

## 3. Neural model

The function `f` is grounded by a neural network.

The model receives the five crab measurements and returns four values in $[0, 1]$, one for each label:

- blue;
- orange;
- male;
- female.

A sigmoid activation is used at the output layer.

In [ ]:
class MLP(torch.nn.Module):
    """Returns logits over 4 labels from 5 input features."""
    def __init__(self, layer_sizes=(5, 16, 16, 8, 4)):
        super().__init__()
        self.elu = torch.nn.ELU()
        self.dropout = torch.nn.Dropout(0.2)
        self.sigmoid = torch.nn.Sigmoid()
        self.linear_layers = torch.nn.ModuleList([
            torch.nn.Linear(layer_sizes[i - 1], layer_sizes[i])
            for i in range(1, len(layer_sizes))
        ])

    def forward(self, x):
        for layer in self.linear_layers[:-1]:
            x = self.dropout(self.elu(layer(x))) if self.training else self.elu(layer(x))
        return self.sigmoid(self.linear_layers[-1](x))

## 4. Interpretation

The interpretation grounds the symbolic vocabulary in concrete computational objects. In this notebook:

- `Item` is interpreted as a five-dimensional tensor type representing crab measurements;
- `Label` is interpreted as a four-dimensional tensor type representing label selectors;
- the function `f` is interpreted as the neural network `MLP`;
- the predicate `P` computes the label-specific truth value by taking the dot product between the network output and a one-hot label vector;
- the four label constants are interpreted as one-hot vectors.

For example, if $f(x)$ returns the four label truth values for sample $x$, then $P(f(x), L_{\mathrm{blue}})$ extracts the truth value assigned to the blue label.

In [ ]:
interp = Interpretation(sig).to(device)

# ground sorts
Crab = sltn.Type("Crab", shape=(5,))
interp["Item"] = Crab

Label    = sltn.Type("Label", shape=(4,))
interp["Label"] = Label

interp["f"] = MLP()
interp["P"] = lambda probs, l: torch.sum(probs * l, dim=-1, keepdim=True)

interp["L_blue"] = torch.tensor([1., 0., 0., 0.])
interp["L_orange"] = torch.tensor([0., 1., 0., 0.])
interp["L_male"] = torch.tensor([0., 0., 1., 0.])
interp["L_female"] = torch.tensor([0., 0., 0., 1.])

print(interp)

## 5. Data loaders and metrics

This section defines the data loader and evaluation utilities.

For each batch, the helper function assigns tensors to the variables declared in the signature:

- `x` receives the full batch;
- `x_blue` receives the blue crabs;
- `x_orange` receives the orange crabs;
- `x_male` receives the male crabs;
- `x_female` receives the female crabs.

The satisfaction metric evaluates the full knowledge base on the current batch assignments. The accuracy metric is computed as a multi-label score. The model predicts four binary labels using a threshold, and the prediction is compared against the one-hot encoding of the true color and sex labels.

In [ ]:
from sklearn.metrics import accuracy_score
from sltn.interpretation import apply_quantifier

import numpy as np

class DataLoader:
    def __init__(self, data, labels, batch_size=64, shuffle=True):
        self.data         = data.to(device)
        self.labels_sex   = labels[0]
        self.labels_color = labels[1]
        self.batch_size   = batch_size
        self.shuffle      = shuffle

    def __len__(self):
        return int(np.ceil(self.data.shape[0] / self.batch_size))

    def __iter__(self):
        idxlist = list(range(self.data.shape[0]))
        if self.shuffle:
            np.random.shuffle(idxlist)
        for start in range(0, self.data.shape[0], self.batch_size):
            idx = idxlist[start:start + self.batch_size]
            yield self.data[idx], self.labels_sex[idx], self.labels_color[idx]

def _set_batch(data, labels_sex, labels_color):
    """Assign current batch slices to the declared variable symbols."""
    interp["x"]       = data
    interp["x_blue"]   = data[labels_color == 0]
    interp["x_orange"] = data[labels_color == 1]
    interp["x_male"]   = data[labels_sex   == 2]
    interp["x_female"] = data[labels_sex   == 3]


def compute_sat_level(loader):
    mean_sat = 0
    for data, lsex, lcolor in loader:
        _set_batch(data, lsex, lcolor)
        mean_sat += interp(phi_kb).value
    return mean_sat / len(loader)

@torch.inference_mode()
def compute_sat_level_phi(loader, inner_phi, p=5):
    """Evaluate ∀ x: inner_phi with strict p-mean-error aggregation (p=5)."""
    strict_interp = interp.with_updates()
    strict_interp.logic = Logic(Forall=agg.AggregPMeanError(p=p))
    mean_sat = 0
    for data, _, _ in loader:
        strict_interp["x"] = data
        mean_sat += apply_quantifier(
            strict_interp.logic.Forall,
            strict_interp(inner_phi),
            ["x"],
        ).value
    return mean_sat / len(loader)


@torch.inference_mode()
def compute_accuracy(loader, threshold=0.5):
    mean_acc = 0.0
    for data, lsex, lcolor in loader:
        interp["x"] = data
        preds = torch.stack([interp(sig.parse("P(f(x), L_blue)")).value, 
                             interp(sig.parse("P(f(x), L_orange)")).value, 
                             interp(sig.parse("P(f(x), L_male)")).value, 
                             interp(sig.parse("P(f(x), L_female)")).value], dim=1).cpu().numpy().squeeze(-1)
        labels_blue   = (lcolor == 0).numpy()
        labels_orange = (lcolor == 1).numpy()
        labels_male   = (lsex   == 2).numpy()
        labels_female = (lsex   == 3).numpy()
        onehot = np.stack([labels_blue, labels_orange, labels_male, labels_female], axis=1)
        pred_binary = (preds > threshold).astype(np.int32)
        hamming = np.count_nonzero(onehot - pred_binary, axis=1) / pred_binary.shape[-1]
        mean_acc += np.mean(1 - hamming)
    return mean_acc / len(loader)


train_loader = DataLoader(train_data, (train_sex_labels, train_color_labels), 64)
test_loader  = DataLoader(test_data,  (test_sex_labels,  test_color_labels),  64, shuffle=False)


## 6. Training loop

Training maximizes the satisfaction of the six-formula knowledge base.

For each batch:

1. the batch is split into the subsets required by the universally quantified formulas;
2. the corresponding variables are grounded in the interpretation;
3. the knowledge base is evaluated under the current neural interpretation;
4. gradients are backpropagated through the fuzzy evaluation of the formulas and through the neural network;
5. the optimizer updates the parameters of `f`.

The query formulas are monitored during training to show how logical consequences and non-consequences behave as the model learns.

In [ ]:
optimizer = torch.optim.Adam(interp["f"].parameters(), lr=0.001)

kb_dict = kb_describe(phi_kb)

for epoch in range(1000):
    train_loss = 0.0
    interp["f"].train()
    for data, lsex, lcolor in train_loader:
        optimizer.zero_grad()
        interp["x"]       = data
        interp["x_blue"]   = data[lcolor == 0]
        interp["x_orange"] = data[lcolor == 1]
        interp["x_male"]   = data[lsex   == 2]
        interp["x_female"] = data[lsex   == 3]
        loss = interp(phi_kb).value
        loss.backward()
        optimizer.step()
        train_loss += loss.item()
    train_loss /= len(train_loader)

    if epoch % 50 == 0:
        interp["f"].eval()
        print(
            f"epoch {epoch:3d} | loss {train_loss:.4f} "
            f"| Train Sat {compute_sat_level(train_loader).item():.3f} "
            f"| Test Sat {compute_sat_level(test_loader).item():.3f} "
            f"| Train Acc {compute_accuracy(train_loader):.3f} "
            f"| Test Acc {compute_accuracy(test_loader):.3f} "
            f"| Sat φ1 {compute_sat_level_phi(test_loader, phi1_inner).item():.3f} "
            f"| Sat φ2 {compute_sat_level_phi(test_loader, phi2_inner).item():.3f} "
            f"| Sat φ3 {compute_sat_level_phi(test_loader, phi3_inner).item():.3f}"
        )

## 7. Query formulas after training

After training, we evaluate the three query formulas on the full test set.

The expected behavior is:

- $\phi_1$: blue implies not orange. This should be highly satisfied because it follows the color mutual-exclusion axiom.
- $\phi_2$: blue implies orange. This should be poorly satisfied because it contradicts the color mutual-exclusion axiom.
- $\phi_3$: blue implies male. This is not constrained by the knowledge base and should generally have no reason to be highly satisfied.

These queries illustrate how trained neural predicates can be inspected using formulas that were not part of the training objective.

In [ ]:
# Set the model to evaluation mode (no dropout)
interp["f"].eval()

sat_phi1 = compute_sat_level_phi(test_loader, phi1_inner).item()
sat_phi2 = compute_sat_level_phi(test_loader, phi2_inner).item()
sat_phi3 = compute_sat_level_phi(test_loader, phi3_inner).item()

print(f"Sat φ1 (blue → ¬orange): {sat_phi1:.3f}")
print(f"Sat φ2 (blue →  orange): {sat_phi2:.3f}")
print(f"Sat φ3 (blue →    male): {sat_phi3:.3f}")